Projekt ma za zadanie zbadać zależność shoalingu (wzrostu amplitudy fali $\zeta_{max}$) od amplitudy początkowej ($A_{initial}$) i wysokości wzniesienia dna ($A_{HILL}$) w modelu Równań Płytkiej Wody (SWE). Układ symulacyjny ma wymiary $100 \times 100 \text{ m}$ z batymetrią ($B_{deep}=1.0 \text{ m}$) i falą Gaussa, propagującą się jednokierunkowo dzięki początkowemu pędowi. Analizowana jest siatka referencyjna $50 \times 50$ z krokiem czasowym dt/dx=0.10.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip --quiet install open-atmos-jupyter-utils
    from open_atmos_jupyter_utils import pip_install_on_colab
    pip_install_on_colab('PyMPDATA-examples')

import numpy as np
from matplotlib import pyplot
from open_atmos_jupyter_utils import show_plot, show_anim
from PyMPDATA import ScalarField, Solver, Stepper, VectorField, Options, boundary_conditions
from IPython.display import display

In [ ]:
class ShallowWaterEquationsIntegrator:
    def __init__(self, *, h_initial: np.ndarray,bathymetry: np.ndarray,uh_initial: np.ndarray, vh_initial: np.ndarray,options: Options = None):
        """ initializes the solvers for a given initial condition of `h` assuming zero momenta at t=0 """
        options = options or Options(nonoscillatory=True, infinite_gauge=True)
        self.bathymetry = bathymetry
        X, Y, grid = 0, 1, h_initial.shape
        stepper = Stepper(options=options, grid=grid)
        kwargs = {
            'boundary_conditions': [
                boundary_conditions.Constant(value=0)] * len(grid),
            'halo': options.n_halo,
        }
        advectees = {
            "h": ScalarField(h_initial, **kwargs),
            "uh": ScalarField(uh_initial, **kwargs),
            "vh": ScalarField(vh_initial, **kwargs),
        }
        self.advector = VectorField((
                np.zeros((grid[X] + 1, grid[Y])),
                np.zeros((grid[X], grid[Y] + 1))
            ), **kwargs
        )
        self.solvers = { k: Solver(stepper, v, self.advector) for k, v in advectees.items() }

    def __getitem__(self, key):
        """ returns `key` advectee field of the current solver state """
        return self.solvers[key].advectee.get()
    
    def _apply_half_rhs(self, *, key, axis, g_times_dt_over_dxy):
        """ applies half of the source term in the given direction """
        self[key][:] -= .5 * g_times_dt_over_dxy * self['h'] * np.gradient(self['h']-self.bathymetry, axis=axis)

    def _update_courant_numbers(self, *, axis, key, mask, dt_over_dxy):
        """ computes the Courant number component from fluid column height and momenta fields """
        velocity = np.where(mask, np.nan, 0)
        momentum = self[key]
        np.divide(momentum, self['h'], where=mask, out=velocity)
        all = slice(None, None) 
        all_but_last = slice(None, -1)
        all_but_first_and_last = slice(1, -1)

        velocity_at_cell_boundaries = velocity[( 
            (all_but_last, all),
            (all, all_but_last),
        )[axis]] + np.diff(velocity, axis=axis) / 2 
        courant_number = self.advector.get_component(axis)[(
            (all_but_first_and_last, all),
            (all, all_but_first_and_last)
        )[axis]]
        courant_number[:] = velocity_at_cell_boundaries * dt_over_dxy[axis]
        assert np.amax(np.abs(courant_number)) <= 1

    def __call__(self, *, nt: int, g: float, dt_over_dxy: tuple, outfreq: int, eps: float=1e-7):
        """ integrates `nt` timesteps and returns a dictionary of solver states recorded every `outfreq` step[s] """
        output = {k: [] for k in self.solvers.keys()}
        for it in range(nt + 1): 
            if it != 0:
                mask = self['h'] > eps
                for axis, key in enumerate(("uh", "vh")):
                    self._update_courant_numbers(axis=axis, key=key, mask=mask, dt_over_dxy=dt_over_dxy)
                self.solvers["h"].advance(n_steps=1)
                for axis, key in enumerate(("uh", "vh")):
                    self._apply_half_rhs(key=key, axis=axis, g_times_dt_over_dxy=g * dt_over_dxy[axis])
                    self.solvers[key].advance(n_steps=1)
                    self._apply_half_rhs(key=key, axis=axis, g_times_dt_over_dxy=g * dt_over_dxy[axis])
            if it % outfreq == 0:
                for key in self.solvers.keys():
                    output[key].append(self[key].copy())
        return output

In [ ]:
g = 10.0
outfreq = 2 
Nx, Ny = 50, 50
Lx, Ly = 100.0, 100.0 
dx = Lx / Nx 
dt_over_dxy = (0.1, 0.1)
dt = dt_over_dxy[0] * dx

B_deep = 1.0        
A_HILL = 0.50       
Sigma_hill = 8.0    
Xc_hill = Lx - Lx / 3.5 
Yc_hill = Ly - Ly / 2 

Amplitude_zeta = 0.40 
L_wave = 10.0
X0_zeta = Lx * 0.15  

Nt_target_time = 17.6
Nt_total = int(Nt_target_time / dt) 

x_coords_phys = np.linspace(0, Lx, Nx)
y_coords_phys = np.linspace(0, Ly, Ny)
X_mesh, Y_mesh = np.meshgrid(x_coords_phys, y_coords_phys, indexing='ij')

def generate_bathymetry(A_hill):
    X_dist_sq = (X_mesh - Xc_hill)**2 + (Y_mesh - Yc_hill)**2 
    return B_deep - A_hill * np.exp(-0.5 * X_dist_sq / Sigma_hill**2)

def generate_initial_h(b_grid_2d):
    gauss_term = np.exp(-0.5 * ((X_mesh - X0_zeta) / L_wave)**2)
    zeta_init = Amplitude_zeta * gauss_term
    return b_grid_2d + zeta_init

bathymetry = generate_bathymetry(A_HILL)
h_initial = generate_initial_h(bathymetry)
zeta_initial = h_initial - bathymetry
C_initial = np.sqrt(g * h_initial)
uh_initial = C_initial * zeta_initial
vh_initial = np.zeros(h_initial.shape, dtype=float)
output = ShallowWaterEquationsIntegrator(
    h_initial=h_initial, bathymetry=bathymetry, uh_initial=uh_initial, vh_initial=vh_initial
)(
    nt=Nt_total, g=g, dt_over_dxy=dt_over_dxy, outfreq=outfreq
)

In [ ]:
Z_MAX = 1
Z_MIN = -1
Z_RANGE = (Z_MIN, Z_MAX)

def plot_wave_propagation_3d(frame, *, zlim=Z_RANGE):
    psi = output['h'][frame]
    zeta = psi - bathymetry
    
    fig, ax = pyplot.subplots(subplot_kw={"projection": "3d"}, figsize=(12, 6))
    ax.plot_wireframe(
        X_mesh, Y_mesh, zeta, 
        color='mediumblue', 
        linewidth=0.75,     
        rstride=2, cstride=2 
    )
    ax.plot_surface(
        X_mesh, Y_mesh, zeta, 
        color='skyblue', 
        alpha=0.3,
        rstride=2, cstride=2, 
        linewidth=0
    )
    pyplot.colorbar(
         ax.contourf(
             X_mesh, Y_mesh, bathymetry, 
             zdir='z', offset=Z_MIN + 0.1, 
             levels=np.linspace(0.5, B_deep, 20),
             cmap=pyplot.cm.viridis
         ),
         pad=.1, aspect=10, fraction=.02, label='batymetria', location='left'
    )
    
    ax.set(
        zlim=zlim, 
        proj_type='ortho', 
        title=f"Propagacja fali | Krok t / Δt = {frame*outfreq} (T={frame*outfreq*dt_over_dxy[0]*dx:.1f}s)", 
        zlabel=r"$\zeta$ [m]"
    )
    ax.set_xlabel(f"x [m]")
    ax.set_ylabel(f"y [m]")

    ax.view_init(elev=20, azim=-120)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.fill = False
        axis.pane.set_edgecolor('black')
        axis.pane.set_alpha(1)
        
    return fig
frame = len(output['h']) - 1
fig = plot_wave_propagation_3d(frame)

show_anim(plot_wave_propagation_3d, range(len(output['h'])))

W celu zachowania poprawności numerycznej nałożono limit czasu na symulację T = 17.6s, aby wyeliminować odbicia od granicznego obszaru. Na animacji prezentującej chwilę końcową (T = 17.6s)widoczny jest wzrost amplitudy fali, która pokonała obszar zmiennej batymetrii. 

In [ ]:
def plot_shoaling_analysis(output, bathymetry, Nt_total, outfreq, dx, dt_over_dxy):
    Ny = bathymetry.shape[1]
    center_y_index = Ny // 2

    x_phys = np.linspace(0, Lx, Nx)

    pyplot.figure(figsize=(12, 6))

    b_section = bathymetry[:, center_y_index]

    pyplot.plot(x_phys, -b_section, 'k--', label='Batymetria -b(x)', alpha=0.7)
    pyplot.fill_between(x_phys, np.amin(-b_section) - 0.1, -b_section, color='saddlebrown', alpha=0.3)

    frame_indices = [
        0,                                   
        Nt_total // (outfreq * 4),           
        Nt_total // (outfreq * 2),           
        len(output['h']) - 1                 
    ]

    colors = ['r', 'orange', 'b', 'darkgreen']
    max_zeta_values = []
    
    for i, frame_index in enumerate(frame_indices):
        zeta_frame = output['h'][frame_index][:, center_y_index] - bathymetry[:, center_y_index]
        current_time = frame_index * outfreq * dt_over_dxy[0] * dx
        max_zeta_values.append(np.amax(zeta_frame))
        
        pyplot.plot(x_phys, zeta_frame, color=colors[i], 
                 label=rf't={current_time:.1f}s ($\zeta_{{max}}={max_zeta_values[-1]:.3f}$m)', 
                 linewidth=1.5)

    pyplot.axhline(0, color='gray', linestyle=':', linewidth=1)
    pyplot.title(rf'Wysokość fali $\zeta(x)$ w czasie')
    pyplot.xlabel('Położenie x [m]')
    pyplot.ylabel(rf'Wysokość $\zeta$ [m]')
    pyplot.legend(loc='upper left')
    pyplot.grid(True, linestyle='--', alpha=0.6)
 
    pyplot.ylim(np.amin(-b_section) - 0.1, np.amax(max_zeta_values) * 1.1)


plot_shoaling_analysis(output, bathymetry, Nt_total, outfreq, dx, dt_over_dxy)

Rysunek 2D ukazuje zmianę amplitudy fali w czasie w kluczowych czterech momentach (przed przejściem, w trakcie przejścia i po przejściu). Przed batymetrią fala utrzymuje stałą amplitudę. Po napotkaniu wzniesienia wysokość impulsu zaczyna rosnąć. Zmiana wysokości fali pokrywa się z oczekiwaniami. Zgodnie ze wzorem $c=\sqrt{hg}$, w miare zbliżania się do wzniesienia $\eta$ wyraźnie wzrasta, a jednocześnie prędkość maleje.

In [ ]:
init_ampl = [0.1, 0.3, 0.5] 
hill_ampl = np.linspace(0.1, 0.8, 7)
all_res = [] 

def generate_initial_h_local(b_grid_2d, A_INIT):
    gauss = np.exp(-0.5 * ((X_mesh - X0_zeta) / L_wave)**2)
    zeta_init = A_INIT * gauss
    return b_grid_2d + zeta_init

for A in init_ampl:
    results_for_one_A = []

    for AH in hill_ampl:
        bathymetry = generate_bathymetry(AH)
        h_initial = generate_initial_h_local(bathymetry, A)
        
        C_initial = np.sqrt(g * h_initial) 
        uh_initial = C_initial * (h_initial - bathymetry)
        vh_initial = np.zeros(h_initial.shape, dtype=float)
  
        output = ShallowWaterEquationsIntegrator(
            h_initial=h_initial, bathymetry=bathymetry,
            uh_initial=uh_initial, vh_initial=vh_initial
        )(
            nt=Nt_total, g=g, dt_over_dxy=dt_over_dxy, outfreq=outfreq
        )

        hmax = np.amax(np.stack(output['h']))
        zeta_max = hmax - np.amin(bathymetry)
        b_min = np.amin(bathymetry)
        
        results_for_one_A.append({
            'A_HILL': AH,
            'b_min': b_min,
            'max_zeta': zeta_max
        })
    all_res.append({
        'A_initial': A,
        'data': results_for_one_A
    })

In [ ]:
pyplot.figure(figsize=(10, 6))

for result_set in all_res:
    init_ampl = result_set['A_initial']
    data = result_set['data']

    b_min_arr = np.array([r['b_min'] for r in data])
    Max_Zeta_data = np.array([r['max_zeta'] for r in data])

    sort_idx = np.argsort(b_min_arr)
    depth = -b_min_arr[sort_idx]

    pyplot.plot(depth, Max_Zeta_data[sort_idx], marker='o',
                label=f'A$_{{initial}}$ = {init_ampl:.2f} m', linewidth=2)

pyplot.title(r'Wpływ batymetrii na $\zeta_{max}$')
pyplot.xlabel('Minimalna głębokość $-b_{min}$ [m]')
pyplot.ylabel(r'Maksymalna zmierzona amplituda $\zeta_{max}$ [m]')
pyplot.grid(True, linestyle='--', alpha=0.6)
pyplot.legend(title='Amplituda Początkowa')
pyplot.tight_layout()

Wykres zależności maksymalnej zmierzonej ampliutudy od batymetrii dla różnych wysokości fali początkowej ($A_{initial}=0.2 \text{ m}, 0.3 \text{ m}, 0.5 \text{ m}$) wizualizuje efekt shoalingu i zależność wzrostu amplitudy od energii fali. Wszystkie trzy krzywe wykazują wyraźny wzrost $\zeta_{max}$ w miarę, jak rośnie $\mathbf{A_{HILL}}$ (czyli maleje głębokość dna $\mathbf{b_{min}}$).

In [ ]:
A_HILL_ref = 0.45

bathymetry_ref = generate_bathymetry(A_HILL_ref)
h_initial_ref = generate_initial_h_local(bathymetry_ref, 0.30)
zeta_initial_ref = h_initial_ref - bathymetry_ref
C_initial_ref = np.sqrt(g * h_initial_ref) 
uh_initial_ref = C_initial_ref * zeta_initial_ref
vh_initial_ref = np.zeros(h_initial_ref.shape, dtype=float)

output_ref = ShallowWaterEquationsIntegrator(
    h_initial=h_initial_ref, bathymetry=bathymetry_ref,
    uh_initial=uh_initial_ref, vh_initial=vh_initial_ref
)(
    nt=Nt_total, g=g, dt_over_dxy=dt_over_dxy, outfreq=outfreq
)

In [ ]:
def plot_velocity_field_anim(frame_index, output, bathymetry, title_suffix):
    h_frame = output['h'][frame_index]
    uh_frame = output['uh'][frame_index]
    vh_frame = output['vh'][frame_index]
    
    eps = 1e-7
    mask = h_frame > eps
    
    u_field = np.where(mask, uh_frame / h_frame, 0)
    v_field = np.where(mask, vh_frame / h_frame, 0)

    skip = 2 
    fig, ax = pyplot.subplots(figsize=(10, 8))

    ax.contourf(X_mesh, Y_mesh, -bathymetry, cmap='viridis', levels=20, alpha=0.6)

    zeta_frame = h_frame - bathymetry
    ax.contour(X_mesh, Y_mesh, zeta_frame, colors='r', levels=[0.05, 0.15, 0.25, 0.35], linewidths=0.5)

    ax.quiver(X_mesh[::skip, ::skip], Y_mesh[::skip, ::skip], 
              u_field[::skip, ::skip], v_field[::skip, ::skip], 
              scale=10,
              color='k', 
              headwidth=5, headlength=5, alpha=0.8)
    
    current_time = frame_index * outfreq * dt_over_dxy[0] * dx
    ax.set_title(f'Pole wektorowe prędkości | T={current_time:.1f}s | {title_suffix}')
    ax.set_xlabel('Położenie x [m]')
    ax.set_ylabel('Położenie y [m]')
    ax.set_aspect('equal', adjustable='box')
    return fig
show_anim(lambda frame: plot_velocity_field_anim(frame, output_ref, bathymetry_ref, f"A_initial={0.40}m, A_HILL={0.50}m"), 
          range(len(output_ref['h'])))

Wizualizacja pola wektorowego prędkości została przeprowadzona dla referencyjnego przypadku $A_{initial}=0.40 \text{ m}$ i góry batymetrycznej $A_{HILL}=0.50 \text{ m}$ ($b_{min}=0.50 \text{ m}$). Wykres przedstawia wektory prędkości cieczy ($\vec{u}$) na tle batymetrii oraz konturów fali ($\zeta$). Wektory prędkości są niemal wyłącznie skierowane w kierunku osi X, co jest dowodem, że inicjalizacja początkowego pędu ($\vec{u}h$) zadziałała poprawnie. W miarę jak fala wchodzi na wzniesienie batymetryczne, długość wektorów maleje (spadek prędkości zgodnie z $c = \sqrt{gh}$), podczas gdy kontury amplitudy ($\zeta$) się zagęszczają (wzrost amplitudy).

In [ ]:
Nx_conv, Ny_conv = 100, 100
X_mesh_conv, Y_mesh_conv = np.meshgrid(np.linspace(0, Lx, Nx_conv), np.linspace(0, Ly, Ny_conv), indexing='ij')

def generate_initial_h_conv(b_grid_2d, A_initial_local):
    gauss = np.exp(-0.5 * ((X_mesh_conv - X0_zeta) / L_wave)**2)
    zeta_init = A_initial_local * gauss
    return b_grid_2d + zeta_init

def generate_bathymetry_conv(A_hill_local):
    X_dist_sq = (X_mesh_conv - Xc_hill)**2 + (Y_mesh_conv - Yc_hill)**2 
    return B_deep - A_hill_local * np.exp(-0.5 * X_dist_sq / Sigma_hill**2)

bathymetry_conv = generate_bathymetry_conv(0.45)
h_initial_conv = generate_initial_h_conv(bathymetry_conv, 0.3)
C_initial_conv = np.sqrt(g * h_initial_conv) 
uh_initial_conv = C_initial_conv * (h_initial_conv - bathymetry_conv)
vh_initial_conv = np.zeros(h_initial_conv.shape, dtype=float)

dx_current = Lx / Nx_conv 
dt_over_dxy = (0.1, 0.1)
dt_conv = dt_over_dxy[0] * dx_current
Nt_target_time_conv = 18.0 
Nt_total_conv = int(Nt_target_time_conv / dt_conv) 

output_conv = ShallowWaterEquationsIntegrator(
    h_initial=h_initial_conv, bathymetry=bathymetry_conv,
    uh_initial=uh_initial_conv, vh_initial=vh_initial_conv
)(
    nt=Nt_total_conv, g=g, dt_over_dxy=dt_over_dxy, outfreq=1 
)
max_h_over_time_conv = np.amax(np.stack(output_conv['h']))
max_zeta_conv = max_h_over_time_conv - np.amin(bathymetry_conv)

zeta_max_45 = 0.949

zeta_max_100 = max_zeta_conv
print("\nTest zbieżności")
print(f"Maksymalna amplituda (50x50, T=18.0s): {zeta_max_45:.4f} m (Referencja)")
print(f"Maksymalna amplituda (100x100, T=18.0s): {zeta_max_100:.4f} m")

diff = np.abs(zeta_max_45 - zeta_max_100)
percent_diff = (diff / zeta_max_45) * 100

print(f"Względna różnica: {percent_diff:.2f} %")


Metodyka badania zbieżności polegała na porównaniu maksymalnej amplitudy fali
 uzyskanej z siatki referencyjnej ($50 \times 50$) z wynikiem siatki 100 \times 100. Zwiększono liczbę kroków, aby zachować ten sam czas symulacji i zapewnić stabilność. Wynik z siatki referencyjnej (0.9490m) różni się o 3.17$\%$ od rezultatu pochodzącego z siatki porównawczej co jest zadowalającym wynikiem. Niepewność pochodzi głównie z niedoskonałości konfiguracji układu (możliwe interferencje pochodzące z drobnych odbić) oraz z dyskretyzacji równań. Ostatnim etapem projektu była próba zbadania warunku załamania fali, która się nie udała. Wprowadzono współczynnik, który mierzy stosunek prędkości wody na grzbiecie $u$ do lokalnej prędkości fali ($c$): $R_u = \frac{u}{c}$. Fala załamuje się, gdy $R_u \ge 1.0$. Pomimo wielu prób i zmian parametrów nie zbliżono się do tej wartości. Powodem były ograniczenia numeryczne związane z warunkiem stabilności CFL > 1.